# Frameworks for Developing LLM Agents

## Course Overview

This course covers the five key techniques for building effective LLM agents using Spring AI:

| # | Module | Topic |
|---|--------|-------|
| 01 | Refining Prompts | Improve output content and format through prompt engineering |
| 02 | RAG Techniques | Augment an agent's context with retrieval-augmented generation |
| 03 | Feedback Loop | Improve agent performance through ongoing feedback |
| 04 | Orchestration | Function calling and multi-agent orchestration |
| 04 | Conversation Memory | Maintain context across multi-turn conversations |

---

# Module 01 — Refining Prompts to Improve Output Content and Format

Prompts are the primary interface between you and an LLM. Small wording changes produce dramatically different outputs. This module focuses on crafting prompts that are clear, context-rich, and output-format-aware.

## 1.1 Prompt Engineering

**Prompt engineering** is the practice of designing, refining, and optimising the text inputs given to a language model in order to control the quality, format, and relevance of its outputs.

It sits at the intersection of:
- **Creativity** (art) — knowing *what* to say and *how* to frame it
- **Precision** (science) — understanding the model's behaviour and constraints

Think of it as programming in natural language: the model is the runtime, and your prompt is the source code.

## 1.2 Prompt Clarity

Ambiguity in language leads to ambiguity in model output. Two classic illustrations:

### Linguistic ambiguity

| Prompt sent | What the model (or reader) infers | What you meant |
|-------------|----------------------------------|----------------|
| `Let's eat Grandma.` | 🍔 ? — is Grandma the meal? | `Let's eat, Grandma.` — an invitation to Grandma |

A missing comma completely changes the meaning. LLMs are sensitive to exactly this kind of ambiguity.

### Mathematical / notational ambiguity

| Prompt sent | What the model infers | What you meant |
|-------------|----------------------|----------------|
| `2 sin((x+y)/2) cos((x+y)/2) …` | `(x+y)/2` ??? | The sum-to-product identity: `x/2 + y/2 = (x+y)/2 …` |

When context is missing, the model guesses — and often guesses wrong.

### Key takeaways

- **Be explicit.** Never rely on implied punctuation, shared background knowledge, or assumed context.
- **Provide the frame.** Tell the model *who* it is, *what* the task is, and *what format* you expect.
- **Iterate.** Treat prompt writing as a feedback loop: observe the output, identify the ambiguity, refine the prompt.

## 1.3 System vs User Prompts in Spring AI

Spring AI's `ChatClient` exposes two prompt layers:

| Layer | Purpose | When it runs |
|-------|---------|-------------|
| **System prompt** | Persona, constraints, output format rules | Once at agent construction (`defaultSystem(...)`) or per-call (`.system(...)`) |
| **User prompt** | The actual question or instruction | Per conversation turn (`.user(...)`) |

### Template parameters

Spring AI supports `{placeholder}` substitution in system prompts:

```java
this.chat = builder
    .defaultSystem("""
        You are a helpful chaperone. Today is {current_date}.
        """)
    .build();

// At call time:
this.chat.prompt()
    .system(s -> s.param("current_date", LocalDate.now().toString()))
    .user(message)
    .call().content();
```

This keeps the *structure* of the prompt stable while injecting dynamic values at runtime.

### Structured output

Instead of parsing free-text responses, use `.entity(MyRecord.class)` to get a typed Java object back:

```java
record Response(String response, List<Activity> activities) {}

Response result = this.chat.prompt()
    .user(message)
    .call().entity(Response.class);
```

Spring AI injects a JSON schema into the prompt automatically so the model returns valid, parseable output.

## 1.4 Demo — Chaperone System Prompt

The following is the system prompt used in the `spring-ai-chaperone` demo project. It demonstrates several best practices:

- **Persona** (`act as if you are a chaperone`)
- **Constraints** (school policy, approved activity list)
- **Context injection** (`{current_date}` template parameter)
- **Decision rules** (what to consider when suggesting activities)
- **Negative constraints** (`IMPORTANT: … DO NOT SUGGEST that activity`)
- **Empathetic tone guidance** (`let them down slowly and with kindness`)

```
Please act as if you are a chaperone for a group of high school students.
They are on a trip and have occasional free time where they will ask
you for suggestions for what to do.

If you don't know the student's name, begin by asking, so that you can
retrieve any previous conversations with them.

To create a list of suggestions, take into account:
  * What day of their trip they are wanting suggestions for
  * What time of day, given the trip itinerary
  * Weather conditions for that time period
  * Activities they've already done or disliked
  * School policy and the pre-approved activity brochure
  * Venue opening hours and travel time
  * Performance days — extra time needed to get ready; stick to nearby activities

IMPORTANT: If the activity cannot be done during their free time (time
constraints, weather, policy, venue closed) DO NOT SUGGEST it.

Today is {current_date}. They've worked hard to get here, help them have fun!
```

---
# Module 02 — Using RAG Techniques to Improve an Agent's Context

LLMs have a fixed training cutoff and a finite context window. **Retrieval-Augmented Generation (RAG)** solves both problems by dynamically fetching relevant documents and injecting them into the prompt at query time.

## 2.1 What is RAG?

RAG is a two-phase pattern:

```
User query
    │
    ▼
[Embed query]  ──▶  Vector similarity search  ──▶  Top-k documents
                                                         │
                              ┌──────────────────────────┘
                              ▼
                   Augmented prompt = system + retrieved docs + user query
                              │
                              ▼
                            LLM  ──▶  Response
```

| Phase | What happens |
|-------|--------------|
| **Indexing** (offline) | Documents are chunked, embedded into vectors, and stored in a vector store |
| **Retrieval** (online) | The user query is embedded; nearest-neighbour search returns the most relevant chunks |
| **Augmentation** | Retrieved chunks are injected into the LLM's context window alongside the query |
| **Generation** | The LLM answers using both its parametric knowledge *and* the retrieved evidence |

## 2.2 RAG in Spring AI

Spring AI's advisor pipeline makes RAG a one-liner:

```java
// 1. Build the vector store (in-memory for demos; swap for Redis/Pinecone/pgvector in prod)
@Bean
VectorStore vectors(EmbeddingModel model) {
    return SimpleVectorStore.builder(model).build();
}

// 2. Index documents at startup
List<Document> docs = new TextReader(resource).read();
docs = new TokenTextSplitter().transform(docs);  // chunk by token count
vectorStore.add(docs);

// 3. Attach the RAG advisor to the ChatClient
this.chat = builder
    .defaultAdvisors(QuestionAnswerAdvisor.builder(vectors).build())
    .build();
```

The `QuestionAnswerAdvisor` intercepts each request, runs a similarity search, and prepends the results to the system prompt automatically.

### Documents loaded in the demo

| File | Content |
|------|---------|
| `trip-itinerary.txt` | Day-by-day schedule: performances, meals, free-time windows |
| `school-policies.txt` | Behavioural rules, prohibited activities, curfews |
| `activity-brochure.txt` | Pre-approved San Francisco activities with hours, cost, and location |

## 2.3 Chunking Strategy

`TokenTextSplitter` is the default chunker in Spring AI. It splits documents into fixed-size token windows with optional overlap.

```
┌─────────────────────────────────────────────┐
│              Full Document                  │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐   │
│  │ Chunk 1  │  │ Chunk 2  │  │ Chunk 3  │   │
│  │ (tokens  │  │ (tokens  │  │ (tokens  │   │
│  │  1-512)  │  │ 449-960) │  │ 897-...) │   │
│  └──────────┘  └──────────┘  └──────────┘   │
│        overlap ───────┘                     │
└─────────────────────────────────────────────┘
```

**Why overlap?** Sentences that span a chunk boundary would otherwise be cut in half. A small overlap ensures context continuity between adjacent chunks.

---
# Module 03 — Improving an Agent's Performance through Feedback

A static system prompt and a fixed document corpus only capture what you knew at build time. **Continuous feedback** lets the agent learn from real user interactions and improve its suggestions over time — without retraining the model.

## 3.1 The Feedback Loop Pattern

```
User interacts with agent
        │
        ▼
Agent suggests Activity X
        │
        ▼
User does Activity X
        │
        ▼
User reports back: "I loved it" / "It was too crowded"
        │
        ▼
Agent calls saveStudentFeedback(feedback)
        │
        ▼
Feedback stored as new Document in VectorStore
        │
        ▼
Next query → RAG retrieves feedback → better suggestions
```

This creates a **self-improving RAG loop**: the vector store grows richer with each interaction, and future similarity searches surface past preferences.

## 3.2 Implementing Feedback with `@Tool` in Spring AI

Spring AI 2.x uses the `@Tool` annotation to expose Java methods as LLM-callable functions:

```java
@Component
public class FeedbackTools {

    private final VectorStore vectors;

    public FeedbackTools(VectorStore vectors) {
        this.vectors = vectors;
    }

    @Tool(description = "Save Student Feedback")
    public void saveStudentFeedback(String feedback) {
        this.vectors.add(List.of(new Document(feedback)));
    }
}
```

The system prompt instructs the agent *when* to call the tool:

```
If you have provided suggestions to them in the past, ask them what
activities they did and how much they enjoyed them. When you receive
feedback, make sure to use the appropriate function to store it and
improve later suggestions.
```

The LLM autonomously decides to call `saveStudentFeedback` — the developer only defines *what* the function does, not *when* to call it.

## 3.3 Feedback vs Fine-tuning

| Approach | Cost | Latency | Personalisation | Update frequency |
|----------|------|---------|-----------------|------------------|
| **RAG feedback** (this module) | Low | Per query | Per user, per session | Real-time |
| **Fine-tuning** | High (GPU time) | Zero at inference | Global model update | Hours to days |
| **RLHF** | Very high | Zero at inference | Global | Weeks |

For most agentic applications, **RAG-based feedback is the right default**: it's cheap, immediate, and naturally scoped to each user's history.

---
# Module 04 — Orchestrating and Conversing with Agents

Real-world agents do more than answer questions — they call external APIs, maintain conversation history, and coordinate with other agents. This module covers **function/tool calling** and **chat memory**.

## 4.1 Orchestrating with Functions (Tool Calling)

Tool calling allows the LLM to request execution of Java methods — enabling access to live data, external APIs, and side effects the model itself cannot perform.

### How it works

```
1. Developer registers tools on the ChatClient
2. Spring AI serialises method signatures + @Tool descriptions → JSON schemas
3. Schemas are included in the LLM request
4. LLM decides: answer directly, OR request tool call
5. If tool call requested → Spring AI executes the method
6. Result is appended to the conversation and sent back to the LLM
7. LLM formulates final answer using the tool result
```

### Registering tools in Spring AI

```java
this.chat = builder
    .defaultTools(weatherTools, feedbackTools)  // pass @Component instances
    .build();
```

### Weather tool example

```java
@Component
public class WeatherTools {

    private final String weatherUrl;

    public WeatherTools(ChaperoneProperties props) {
        this.weatherUrl = props.weatherUrl();
    }

    @Tool(description = "Get Weather Forecast")
    public List<WeatherResponse> getWeatherForecast() {
        RestTemplate rest = new RestTemplate();
        return rest.getForObject(URI.create(weatherUrl), Response.class)
                   .properties().periods();
    }

    // Records matching NWS API response shape
    public record WeatherResponse(
        ZonedDateTime startTime, ZonedDateTime endTime,
        Integer temperature, String windSpeed,
        Precipitation probabilityOfPrecipitation,
        String shortForecast) {}
}
```

The agent automatically calls `getWeatherForecast()` whenever it needs current conditions — the developer never hard-codes *when* to call it.

## 4.2 Conversing with Memory

By default, each LLM call is stateless — the model has no memory of prior turns. Spring AI's `MessageChatMemoryAdvisor` fixes this by injecting conversation history into every request.

### Memory setup

```java
// Define the memory store bean
@Bean
ChatMemory memory() {
    return MessageWindowChatMemory.builder().build(); // keeps last N messages
}

// Attach to ChatClient
this.chat = builder
    .defaultAdvisors(
        MessageChatMemoryAdvisor.builder(memory).build(),
        QuestionAnswerAdvisor.builder(vectors).build()
    )
    .build();
```

### Per-conversation isolation

Each student gets their own conversation thread via a UUID chat ID:

```java
private final Map<String, String> studentChats = new ConcurrentHashMap<>();

public String chat(String chatId, String message) {
    return this.chat.prompt()
        .system(s -> s.param("current_date", LocalDate.now().toString()))
        .user(message)
        .advisors(a -> a.param(ChatMemory.CONVERSATION_ID, chatId))
        .call().content();
}

@PostMapping("/chat")
public Map<String, String> chatEndpoint(@RequestBody Map<String, String> body) {
    String studentName = body.get("studentName");
    String message     = body.get("message");
    String chatId = studentChats.computeIfAbsent(studentName,
                                                  n -> UUID.randomUUID().toString());
    return Map.of("response", chat(chatId, message));
}
```

### Memory strategies

| Strategy | Implementation | Trade-off |
|----------|---------------|----------|
| **Message window** | `MessageWindowChatMemory` | Simple; oldest messages evicted when window full |
| **Token window** | custom | More precise; avoids cutting mid-conversation |
| **Summarisation** | call LLM to compress history | Longest effective memory; extra latency + cost |
| **External store** | Redis / database | Survives restarts; required for production |

## 4.3 The Spring AI Advisor Pipeline

All the capabilities above compose through the **advisor chain** — middleware that wraps every `ChatClient` call:

```
User message
     │
     ▼
┌─────────────────────────┐
│  MessageChatMemoryAdvisor│  ← injects prior conversation turns
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  QuestionAnswerAdvisor  │  ← injects RAG document chunks
└────────────┬────────────┘
             │
             ▼
         LLM call  ──▶  (optional tool calls)  ──▶  Final response
```

Advisors are applied in registration order on the way **in**, and in reverse order on the way **out** — similar to servlet filters or Spring Security's filter chain.

---
# Course Summary

| Technique | What it solves | Spring AI API |
|-----------|---------------|---------------|
| **Refine Your Prompts** | Vague, inconsistent, or wrongly-formatted output | `defaultSystem()`, `.system(s -> s.param(...))`, `.entity(Class)` |
| **Use RAG Techniques** | Model lacks domain knowledge or recent information | `SimpleVectorStore`, `TokenTextSplitter`, `QuestionAnswerAdvisor` |
| **Provide Ongoing Feedback** | Suggestions don't improve over time | `@Tool saveStudentFeedback()`, `VectorStore.add()` |
| **Orchestrate with Functions** | Agent needs live data or must trigger side effects | `@Tool`, `defaultTools()`, automatic tool-call loop |
| **Converse with Memory** | Agent forgets previous messages | `MessageWindowChatMemory`, `MessageChatMemoryAdvisor`, `CONVERSATION_ID` |

These five techniques are **composable** — the demo project (`spring-ai-chaperone`) uses all of them together.